# DATA PIPELINE & EXPLORATORY DATA ANALYSIS (EDA)
**Objective:** Load raw data, perform initial exploratory data analysis, generate mock data for prototyping, and establish the PyTorch DataLoader pipeline.

## 1. Import Libraries & Setup Directories

## 2. Load Raw Data & Extract Mock Data
Generate a small random subset (5,000 samples) to unblock the Modeling team for training loop construction.

In [1]:
import os
import pandas as pd  

# Create directory structure if it doesn't exist
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True) 

# Load the original massive dataset (Replace with your actual file path)
# Assuming the file has 'text' and 'label' columns
df_raw = pd.read_csv(r"C:\nlp_project\data\raw\text.csv") 

# Randomly sample 5,000 rows for the team to construct the initial ML pipeline
df_mock = df_raw.sample(n=5000, random_state=42)

# Save the mock dataset to a separate CSV file 
df_mock.to_csv(r"C:\nlp_project\data\raw\mock_data.csv", index=False) 
print(f"Mock dataset successfully generated with {len(df_mock)} rows.") 

Mock dataset successfully generated with 5000 rows.


## 3. Exploratory Data Analysis (EDA)
Analyze class distribution (imbalance check) and sequence lengths to determine optimal modeling parameters.

In [2]:
# Check total data distribution across the 6 emotion labels
label_distribution = df_raw['label'].value_counts()
label_percentage = df_raw['label'].value_counts(normalize=True) * 100

print("--- Class Distribution ---")
for label, count in label_distribution.items():
    percentage = label_percentage[label]
    print(f"Label {label}: {count} samples ({percentage:.2f}%)")

# Calculate text length (word count) to help determine the optimal max_length for BERT
df_raw['word_count'] = df_raw['text'].apply(lambda x: len(str(x).split()))
max_words = df_raw['word_count'].max()
avg_words = df_raw['word_count'].mean()

print("\n--- Text Length Analysis ---")
print(f"Maximum sequence length: {max_words} words") 
print(f"Average sequence length: {avg_words:.2f} words") 

--- Class Distribution ---
Label 1: 141067 samples (33.84%)
Label 0: 121187 samples (29.07%)
Label 3: 57317 samples (13.75%)
Label 4: 47712 samples (11.45%)
Label 2: 34554 samples (8.29%)
Label 5: 14972 samples (3.59%)

--- Text Length Analysis ---
Maximum sequence length: 178 words
Average sequence length: 19.21 words


## 4. Build PyTorch Dataset & DataLoader Prototype
Construct a custom PyTorch `Dataset` to handle text sequences and label encoding. This prototype establishes an optimized data pipeline to feed iterable tensor batches into the neural network architectures (e.g., LSTM, Transformers). We will validate the tensor dimensions using the generated mock data to effectively unblock the Modeling team.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader 

class EmotionDataset(Dataset):
    def __init__(self, dataframe):
        """
        Custom PyTorch Dataset for managing text data and corresponding emotion labels.
        """
        self.texts = dataframe['text'].values
        self.labels = dataframe['label'].values

    def __len__(self):
        # Return the total number of samples
        return len(self.labels)

    def __getitem__(self, idx):
        # Fetch a single data point by its index
        text = str(self.texts[idx])
        label = self.labels[idx] 
        
        # Labels must be converted to long tensors for PyTorch CrossEntropyLoss
        return text, torch.tensor(label, dtype=torch.long)

# Test the pipeline using the mock data generated earlier
mock_dataset = EmotionDataset(df_mock)

# Define the Batch Size configuration
BATCH_SIZE = 32

# Construct PyTorch DataLoader
mock_loader = DataLoader(mock_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("\n--- DataLoader Verification (Mock Data) ---")
# Extract a single sample batch to verify the pipeline logic
sample_texts, sample_labels = next(iter(mock_loader))
print(f"Batch text tensor shape (tuple size): {len(sample_texts)}")
print(f"Batch label tensor shape           : {sample_labels.shape}")
print(f"First text sample in batch         : '{sample_texts[0]}'")
print(f"First label sample in batch        : {sample_labels[0].item()}") 

NameError: name 'Dataset' is not defined